In [1]:
"""
Financing cash-flow breakdown of a delta-hedged option.

Setup: we SELL 1 call (100 shares) and delta-hedge daily until expiry.
Every day the desk's cash account moves for four distinct reasons:

    1. share trades        -(dDelta) * S        (rebalancing)
    2. interest on cash     B * (e^{r dt} - 1)
    3. dividends received   Delta * S * (e^{q dt} - 1)
    4. option payoff        at expiry only

The point of this script is to split (2) + (3) -- the *financing* leg --
into the two economically meaningful pieces:

    interest on free cash   = (B + Delta*S) * (e^{r dt} - 1)   ~  +r * C  * dt
    carry on the hedge      = -Delta*S*(e^{r dt}-1) + divs     ~  -(r-q) * Delta*S * dt

Those are exactly the two rate terms of the Black-Scholes PDE:

    Theta + 0.5*sigma^2*S^2*Gamma + (r-q)*S*Delta - r*C = 0
                                    ^^^^^^^^^^^^^^^^^^^
                                    the financing leg
"""

import math
import random

# ----------------------------------------------------------------------
# Black-Scholes
# ----------------------------------------------------------------------

def _N(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_call(S, K, r, q, sigma, T):
    """Return (price, delta, gamma) of a European call."""
    if T <= 0:
        intrinsic = max(S - K, 0.0)
        return intrinsic, (1.0 if S > K else 0.0), 0.0
    vol = sigma * math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / vol
    d2 = d1 - vol
    df_q, df_r = math.exp(-q * T), math.exp(-r * T)
    price = S * df_q * _N(d1) - K * df_r * _N(d2)
    delta = df_q * _N(d1)
    pdf = math.exp(-0.5 * d1 * d1) / math.sqrt(2 * math.pi)
    gamma = df_q * pdf / (S * vol)
    return price, delta, gamma


# ----------------------------------------------------------------------
# Market / trade parameters
# ----------------------------------------------------------------------

S0        = 100.0     # spot
K         = 100.0     # strike
R         = 0.05      # funding / discount rate (continuous)
Q         = 0.02      # dividend yield (continuous)
SIG_IMP   = 0.20      # implied vol we sold at
SIG_REAL  = 0.25      # vol the stock actually realises
N_DAYS    = 21        # 1 month, hedged once a day
DT        = 1.0 / 252
NOTIONAL  = 100       # shares per contract; we are SHORT 1 contract
SEED      = 7


def simulate_path(seed=SEED, sigma=SIG_REAL):
    rng = random.Random(seed)
    path, S = [S0], S0
    for _ in range(N_DAYS):
        z = rng.gauss(0.0, 1.0)
        S *= math.exp((R - Q - 0.5 * sigma ** 2) * DT + sigma * math.sqrt(DT) * z)
        path.append(S)
    return path


# ----------------------------------------------------------------------
# The hedging ledger
# ----------------------------------------------------------------------

def run_hedge(path, verbose=True):
    n = NOTIONAL
    T = N_DAYS * DT

    # --- day 0: sell the option, buy the initial hedge -----------------
    C0, d0, _ = bs_call(S0, K, R, Q, SIG_IMP, T)
    premium = C0 * n                 # cash IN  (we are short the option)
    shares = d0 * n
    cash = premium - shares * S0     # cash OUT to buy the hedge -> usually negative

    rows, fin_rows = [], []
    tot_int_free = tot_carry = tot_divs = 0.0
    tot_trade_pnl = 0.0

    for i in range(N_DAYS):
        S, tau = path[i], T - i * DT
        free_cash = cash + shares * S          # = premium + accumulated P&L

        # ---- financing accrued over the day ---------------------------
        gross_int = cash * (math.exp(R * DT) - 1.0)          # on the raw balance
        divs = shares * S * (math.exp(Q * DT) - 1.0)         # on the shares held

        # same number, split into its two economic pieces:
        int_free = free_cash * (math.exp(R * DT) - 1.0)      # +r*C  : we earn on the premium
        carry = -shares * S * (math.exp(R * DT) - 1.0) + divs  # -(r-q)*S*Delta : cost of the hedge
        assert abs((gross_int + divs) - (int_free + carry)) < 1e-9

        cash += gross_int + divs
        tot_int_free += int_free
        tot_carry += carry
        tot_divs += divs

        # ---- mark to the next day and rebalance -----------------------
        S_next, tau_next = path[i + 1], tau - DT
        tot_trade_pnl += shares * (S_next - S)

        _, d_next, _ = bs_call(S_next, K, R, Q, SIG_IMP, tau_next) if tau_next > 0 else (0, 0, 0)
        new_shares = d_next * n if tau_next > 0 else 0.0     # liquidate at expiry
        traded = new_shares - shares
        cash -= traded * S_next
        shares = new_shares

        rows.append((i + 1, S_next, d_next, traded, -traded * S_next, cash))
        fin_rows.append((i + 1, cash, shares * S_next, free_cash,
                         int_free, carry, divs, int_free + carry))

    # --- expiry --------------------------------------------------------
    S_T = path[-1]
    payoff = max(S_T - K, 0.0) * n      # we PAY this (short call)
    cash -= payoff
    pnl = cash                          # no shares left, so cash == total P&L

    # independent check of the same P&L
    check = premium + tot_trade_pnl + tot_int_free + tot_carry - payoff
    assert abs(check - pnl) < 1e-8, (check, pnl)

    if verbose:
        print(f"Short 1 call, K={K:.0f}, {N_DAYS} business days, sold at "
              f"{SIG_IMP:.0%} implied vs {SIG_REAL:.0%} realised")
        print(f"r = {R:.2%}, q = {Q:.2%}, notional = {NOTIONAL} shares\n")
        print(f"  premium received : {premium:10.2f}   (delta0 = {d0:.3f}, "
              f"hedge cost = {d0 * n * S0:,.2f})")
        print(f"  opening cash     : {premium - d0 * n * S0:10.2f}   "
              f"<- negative means we BORROW to carry the hedge\n")

        print("Daily hedging ledger")
        print(f"{'day':>4}{'S':>9}{'delta':>8}{'d(sh)':>9}{'trade CF':>11}{'cash bal':>12}")
        for d, S, dl, tr, cf, cb in rows:
            print(f"{d:>4}{S:>9.2f}{dl:>8.3f}{tr:>9.2f}{cf:>11.2f}{cb:>12.2f}")

        print("\nFinancing breakdown  (interest on free cash vs carry on the hedge)")
        print(f"{'day':>4}{'cash B':>12}{'stock S*Δ':>12}{'free cash':>11}"
              f"{'int free':>10}{'carry':>9}{'(of which div)':>15}{'net fin':>9}")
        for d, cb, sv, fc, itf, cr, dv, net in fin_rows:
            print(f"{d:>4}{cb:>12.2f}{sv:>12.2f}{fc:>11.2f}"
                  f"{itf:>10.4f}{cr:>9.4f}{dv:>15.4f}{net:>9.4f}")

        print("\nP&L attribution")
        print(f"  option premium received        {premium:>10.2f}")
        print(f"  hedge (spot) P&L               {tot_trade_pnl:>10.2f}")
        print(f"  interest on free cash  (+rC)   {tot_int_free:>10.2f}")
        print(f"  carry on hedge     (-(r-q)SΔ)  {tot_carry:>10.2f}")
        print(f"      of which dividends         {tot_divs:>10.2f}")
        print(f"  -------------------------------{'':->10}")
        print(f"  NET FINANCING                  {tot_int_free + tot_carry:>10.2f}")
        print(f"  terminal payoff paid           {-payoff:>10.2f}")
        print(f"  ===============================" + "=" * 10)
        print(f"  TOTAL P&L                      {pnl:>10.2f}\n")

        # sanity: theoretical financing over the life
        approx = sum((R * bs_call(path[i], K, R, Q, SIG_IMP, T - i * DT)[0]
                      - (R - Q) * path[i] * bs_call(path[i], K, R, Q, SIG_IMP, T - i * DT)[1])
                     * n * DT for i in range(N_DAYS))
        print(f"  PDE check  Σ[rC - (r-q)SΔ]dt =  {approx:>10.2f}   "
              f"(vs {tot_int_free + tot_carry:.2f} realised)")

    return pnl


# ----------------------------------------------------------------------
# Validation: if realised vol == implied, the hedge should break even
# ----------------------------------------------------------------------

def monte_carlo(sigma_real, n_paths=3000):
    global SIG_REAL
    keep, SIG_REAL = SIG_REAL, sigma_real
    tot = 0.0
    for s in range(n_paths):
        tot += run_hedge(simulate_path(seed=1000 + s, sigma=sigma_real), verbose=False)
    SIG_REAL = keep
    return tot / n_paths


if __name__ == "__main__":
    run_hedge(simulate_path())

    print("\nMonte-Carlo of the same strategy (3000 paths, mean P&L):")
    for sr in (0.15, 0.20, 0.25):
        tag = "  <- sold at fair vol, breaks even" if abs(sr - SIG_IMP) < 1e-9 else ""
        print(f"  realised vol {sr:>5.0%} : {monte_carlo(sr):>9.2f}{tag}")
    print("\nBreak-even at realised == implied is the proof that the financing"
          "\nleg above is booked correctly: get the carry wrong and this drifts.")

Short 1 call, K=100, 21 business days, sold at 20% implied vs 25% realised
r = 5.00%, q = 2.00%, notional = 100 shares

  premium received :     242.31   (delta0 = 0.528, hedge cost = 5,278.86)
  opening cash     :   -5036.55   <- negative means we BORROW to carry the hedge

Daily hedging ledger
 day        S   delta    d(sh)   trade CF    cash bal
   1    99.60   0.499    -2.91     290.31    -4746.82
   2   100.40   0.556     5.68    -570.14    -5317.51
   3   100.04   0.529    -2.63     263.13    -5054.99
   4    99.55   0.491    -3.87     385.01    -4670.56
   5    98.10   0.375   -11.55    1133.52    -3537.58
   6    97.77   0.344    -3.11     304.33    -3233.66
   7    99.50   0.480    13.66   -1358.78    -4592.81
   8   100.16   0.537     5.61    -561.70    -5155.05
   9   101.81   0.679    14.24   -1449.35    -6604.99
  10   102.21   0.717     3.82    -390.51    -6996.26
  11   102.85   0.774     5.72    -588.41    -7585.48
  12   103.15   0.807     3.24    -333.97    -7920.32
 